In [1]:

!mamba install pandas matplotlib seaborn scikit-learn
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import randint

# Load data
df = pd.read_csv("qs-world-rankings-2025-cleaned.csv")

# Handle either schema
rank_2025_col = "2025 Rank Numeric" if "2025 Rank Numeric" in df.columns else "2025 Rank"
rank_2024_col = "2024 Rank Numeric" if "2024 Rank Numeric" in df.columns else "2024 Rank"

# Feature engineering (same style as your file)
df["Rank Movement"] = df[rank_2024_col] - df[rank_2025_col]
df["Reputation Score"] = df[["Academic Reputation", "Employer Reputation"]].mean(axis=1)
df["Research Strength"] = df[["Citations per Faculty", "International Research Network"]].mean(axis=1)
df["Internationalization Score"] = df[["International Faculty", "International Students", "International Research Network"]].mean(axis=1)
df["Career Outcomes Score"] = df[["Employer Reputation", "Employment Outcomes"]].mean(axis=1)
df["Student Support Score"] = df[["Faculty Student", "International Students"]].mean(axis=1)
df["International Balance Gap"] = (df["International Faculty"] - df["International Students"]).abs()
df["Reputation Gap"] = df["Academic Reputation"] - df["Employer Reputation"]

# Binary target
df["Top 50 Flag"] = (df[rank_2025_col] <= 50).astype(int)

features = [
    "Reputation Score",
    "Research Strength",
    "Internationalization Score",
    "Career Outcomes Score",
    "Student Support Score",
    "International Balance Gap",
    "Reputation Gap",
]

X = df[features]
y = df["Top 50 Flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# -------------------------
# 1) Logistic Regression + GridSearchCV
# -------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_grid = GridSearchCV(
    estimator=LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42),
    param_grid={
        "C": [0.01, 0.1, 1, 10, 100],
        "solver": ["liblinear", "lbfgs"],
    },
    scoring="f1",   # better than accuracy for imbalanced classes
    cv=cv,
    n_jobs=-1,
)
log_grid.fit(X_train_scaled, y_train)

best_log = log_grid.best_estimator_
log_train_pred = best_log.predict(X_train_scaled)
log_test_pred = best_log.predict(X_test_scaled)

print("\n=== Logistic Regression (GridSearchCV) ===")
print("Best params:", log_grid.best_params_)
print("Best CV F1:", round(log_grid.best_score_, 4))
print("Train accuracy:", round(accuracy_score(y_train, log_train_pred), 4))
print("Test accuracy:", round(accuracy_score(y_test, log_test_pred), 4))
print(classification_report(y_test, log_test_pred, digits=4))


# 2) Random Forest + RandomizedSearchCV

rf_random = RandomizedSearchCV(
    estimator=RandomForestClassifier(class_weight="balanced", random_state=42),
    param_distributions={
        "n_estimators": randint(100, 500),
        "max_depth": [None, 8, 10, 12, 15, 20],
        "min_samples_split": randint(2, 15),
        "min_samples_leaf": randint(1, 6),
        "max_features": ["sqrt", "log2"],
    },
    n_iter=40,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
)
rf_random.fit(X_train, y_train)

best_rf_random = rf_random.best_estimator_
rf_train_pred = best_rf_random.predict(X_train)
rf_test_pred = best_rf_random.predict(X_test)

print("\n=== Random Forest (RandomizedSearchCV) ===")
print("Best params:", rf_random.best_params_)
print("Best CV F1:", round(rf_random.best_score_, 4))
print("Train accuracy:", round(accuracy_score(y_train, rf_train_pred), 4))
print("Test accuracy:", round(accuracy_score(y_test, rf_test_pred), 4))
print(classification_report(y_test, rf_test_pred, digits=4))




mambajs 0.19.13

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, seaborn, scikit-learn
Channels: emscripten-forge, conda-forge

Solving environment...
Solving took 1.0442000000476837 seconds
  Name                          Version                       Build                         Channel                       
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
+ brotli-python                 1.2.0                         py313h33caa6c_0               emscripten-forge              
+ certifi                       2026.2.25                     pyhd8ed1ab_0                  conda-forge                   
+ charset-normalizer            3.4.7                         pyhd8ed1ab_0                  conda-forge                   
+ idna                          3.11                          pyhd8ed1ab_0                  conda-forge                   
+ joblib                    